# Zbieranie korpusu kolokacji — Colab GPU

Parsuje polskie teksty Stanzą i zapisuje trójki kolokacyjne do pliku.

**Zanim uruchomisz:** Środowisko wykonawcze → Zmień typ środowiska → **GPU**.

## Jak zmienić skalę przebiegu

Jedno miejsce: w **komórce 4** ustaw `SKALA`.

| `SKALA` | tokenów | czas na T4 | Dysk | baza |
|---|---:|---:|---|---:|
| `"proba"` | 5 mln | ~2 h | nie | ~39 MB |
| `"pelna"` | 30 mln | ~12 h | tak, automatycznie | ~187 MB |

Budżet, nazwy plików i zapis na Dysk wynikają z tej jednej zmiennej — nie
trzeba nic edytować w dalszych komórkach.

Na kartach L4 lub A100 (Colab Pro) będzie **znacznie szybciej** niż podane
wyżej wartości dla T4. Komórka **4b** zmierzy to na Twojej karcie i poda
konkretną liczbę godzin.

## Kolejność

Uruchamiaj po kolei. Komórka **3b** sprawdza środowisko — jeśli coś się nie
doinstalowało albo klon repozytorium padł, dowiesz się tam, a nie w połowie
zbierania. Komórka **4b** dobiera rozmiar partii sieci; przy pełnej skali
warto ją uruchomić, bo to jedyny parametr realnie wpływający na tempo.

Kod pochodzi z repozytorium projektu, a nie jest wklejony do notebooka — celowo.
Ekstraktor trójek musi być **dokładnie ten sam**, którego użyje dodatek w czasie
pisania. Rozjazd logiki klucza między budową bazy a zapytaniem nie objawiłby się
żadnym błędem — tylko cicho zerową skutecznością podpowiedzi.

In [ ]:
# 1. Kontrola GPU — bez niego przebieg potrwa kilkanaście razy dłużej
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "BRAK GPU — zmień typ środowiska wykonawczego"

In [ ]:
# 2. Kod projektu + zależności
#
# Repozytorium jest publiczne, więc klon idzie anonimowo i nic nie trzeba
# konfigurować.
#
# Gdyby kiedyś wróciło do prywatnego: utwórz fine-grained token na GitHubie
# (Settings → Developer settings → Personal access tokens; dostęp tylko do tego
# repo, uprawnienie Contents: Read-only), a w Colabie dodaj go pod ikoną klucza
# (🔑) jako sekret o nazwie GH_TOKEN. Kod poniżej wykryje go sam.

WLASCICIEL = "SernikNaZimno"
NAZWA_REPO = "Autokorekta-Kolokacji"

import os, shutil, subprocess

token = None
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    pass  # brak sekretu — repo jest publiczne, więc to normalna ścieżka

url = (
    f"https://{token}@github.com/{WLASCICIEL}/{NAZWA_REPO}.git"
    if token
    else f"https://github.com/{WLASCICIEL}/{NAZWA_REPO}.git"
)

if os.path.exists("/content/projekt"):
    shutil.rmtree("/content/projekt")

# Bez tego git czeka na hasło w nieskończoność zamiast zwrócić błąd.
os.environ["GIT_TERMINAL_PROMPT"] = "0"
wynik = subprocess.run(
    ["git", "clone", "--depth", "1", url, "/content/projekt"],
    capture_output=True, text=True,
)

# Zatrzymujemy się TUTAJ, jeśli klon padł. Wcześniejsza wersja leciała dalej,
# przez co brak repozytorium ujawniał się dopiero jako „No module named
# 'backend'" sześć komórek później i wyglądał na błąd kodu, a nie dostępu.
if wynik.returncode != 0:
    komunikat = wynik.stderr.replace(token, "***") if token else wynik.stderr
    raise RuntimeError(
        "KLON NIEUDANY — notebook zatrzymany celowo.\n\n"
        + komunikat.strip()
        + "\n\nJeśli widzisz 'could not read Password' lub 'Repository not found':\n"
        "repozytorium nie jest publiczne. Sprawdź GitHub → Settings → General →\n"
        "Change repository visibility, albo dodaj sekret GH_TOKEN (patrz góra komórki)."
    )

%cd /content/projekt
print("sklonowano OK")
!pip install -q stanza datasets morfeusz2

In [ ]:
# 3. Model polski (pobiera się raz, ~500 MB)
import stanza
stanza.download("pl", verbose=False)
print("model gotowy")

In [ ]:
# 3b. KONTROLA ŚRODOWISKA — uruchom przed zbieraniem
#
# `pip install -q` NIE przerywa notebooka, gdy instalacja się nie uda. Bez tej
# komórki brakujący pakiet objawiłby się dopiero w środku zbierania i wyglądał
# na błąd pipeline'u, a nie środowiska.

import sys, os

KORZEN = "/content/projekt"   # ścieżka bezwzględna: nie zależy od katalogu roboczego
if KORZEN not in sys.path:
    sys.path.insert(0, KORZEN)

problemy = []

if not os.path.isdir(os.path.join(KORZEN, "backend")):
    problemy.append(f"brak {KORZEN}/backend — klon repozytorium się nie powiódł (komórka 2)")

try:
    import torch
    print(f"torch {torch.__version__}   CUDA dostępna: {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        problemy.append("BRAK GPU — Środowisko wykonawcze → Zmień typ środowiska → T4")
except Exception as e:
    problemy.append(f"torch: {e}")

for nazwa in ["stanza", "datasets", "morfeusz2"]:
    try:
        __import__(nazwa)
        print(f"{nazwa}: OK")
    except Exception as e:
        problemy.append(f"{nazwa}: {e}")

for modul in ["backend.ekstraktor", "backend.baza", "backend.czyszczenie",
              "backend.slownik", "backend.pipeline"]:
    try:
        __import__(modul)
        print(f"{modul}: OK")
    except Exception as e:
        problemy.append(f"{modul}: {e}")

# Morfeusz musi nie tylko się zaimportować, ale i mieć wczytany słownik
try:
    from backend.slownik import WalidatorSlownikowy
    w = WalidatorSlownikowy()
    assert w.znane("decyzja") and not w.znane("złdo")
    print("filtr słownikowy: OK")
except Exception as e:
    problemy.append(f"filtr słownikowy: {e}")

print()
if problemy:
    print("PROBLEMY — napraw przed dalszymi krokami:")
    for p in problemy:
        print(f"  · {p}")
else:
    print("Wszystko gotowe.")

In [ ]:
# 4. KONFIGURACJA PRZEBIEGU — jedyne miejsce, które zmieniasz
#
#   "proba"  —  5 mln tokenów, ~2 h na T4. Sprawdzenie, czy wszystko działa.
#   "pelna"  — 30 mln tokenów, ~12 h na T4 (znacznie mniej na L4/A100).
#              Z tej bazy korzysta docelowy dodatek.
#
# Proporcja 60/40 między Wikipedią a webem jest celowa: w przebiegu na 5 mln
# dała faktyczne 64,8% / 35,2%. Sama Wikipedia narzuciłaby rejestr
# encyklopedyczny, a sam web — szablony sklepowe. Oba źródła wnoszą coś,
# czego drugie nie ma.

SKALA = "proba"          # <<< TU zmieniasz: "proba" albo "pelna"

USTAWIENIA = {
    "proba": {"wiki":  3_000_000, "web":  2_000_000, "dysk": False, "etykieta": "5M"},
    "pelna": {"wiki": 18_000_000, "web": 12_000_000, "dysk": True,  "etykieta": "30M"},
}
_k = USTAWIENIA[SKALA]

BUDZET = {"wiki": _k["wiki"], "web": _k["web"]}
UZYJ_DYSKU = _k["dysk"]

# Rozmiar partii sieci. None = domyślne Stanzy; komórka 4b nadpisze zmierzoną
# wartością. NIE ustawiaj tu na oko — wcześniejsze 256 było MNIEJSZE niż
# domyślne 400 i spowalniało przebieg.
PARTIA_MODELU = None

from pathlib import Path

# Przy pełnej skali Dysk jest włączony automatycznie: kilkunastogodzinny
# przebieg zapisany w /content zniknąłby razem z sesją.
if UZYJ_DYSKU:
    from google.colab import drive
    drive.mount("/content/drive")
    KATALOG = Path("/content/drive/MyDrive/kolokacje")
else:
    KATALOG = Path("/content/wyniki")

KATALOG.mkdir(parents=True, exist_ok=True)

WYJSCIE = KATALOG / f"trojki_{_k['etykieta']}.tsv.gz"
BAZA = KATALOG / f"kolokacje_{_k['etykieta']}.sqlite"

print(f"skala:   {SKALA}  —  {sum(BUDZET.values()):,} tokenów".replace(",", " "))
print(f"budżet:  wiki {BUDZET['wiki']:,}, web {BUDZET['web']:,}".replace(",", " "))
print(f"zapis:   {WYJSCIE}")
print(f"Dysk:    {'TAK' if UZYJ_DYSKU else 'nie (pobierz plik komórką 8!)'}")

In [ ]:
# 4b. POMIAR PRZEPUSTOWOŚCI — uruchom PRZED pełną skalą
#
# Rozkład czasu zmierzony osobno (scripts/diagnoza_przepustowosci.py):
#   parsowanie Stanzą  94%
#   pobieranie + filtr  6%
#   filtr Morfeusza     0%
# Wszystko poza Stanzą to walka o najwyżej 6%. Jedyne, co realnie zmienia
# tempo, to rozmiar partii sieci — a jego optimum zależy od karty i długości
# zdań, więc trzeba je zmierzyć, nie zgadnąć.
#
# Ta komórka USTAWIA zmienną PARTIA_MODELU, z której skorzysta komórka 5.
# Dwie minuty tutaj mogą oszczędzić kilka godzin.

import sys, time
if "/content/projekt" not in sys.path:
    sys.path.insert(0, "/content/projekt")

from backend.pipeline import parsuj_do_trojek, utworz_pipeline, zdania_ze_zrodla

print("pobieram próbkę...")
PROBKA = list(zdania_ze_zrodla("wiki", 25_000))
TOKENOW = sum(len(z.split()) for z in PROBKA)
CEL = sum(BUDZET.values())
print(f"{len(PROBKA)} zdań, {TOKENOW:,} tokenów\n")

print(f"{'partia sieci':>13} {'tok/s':>9} {'Twój przebieg':>16}")
print("-" * 42)
wyniki = {}
for partia in [None, 400, 1000, 2000]:
    try:
        nlp = utworz_pipeline(gpu=True, partia=partia)
        t = time.perf_counter()
        sum(1 for _ in parsuj_do_trojek(PROBKA, nlp, "wiki", partia=256))
        tps = TOKENOW / (time.perf_counter() - t)
        wyniki[partia] = tps
        etykieta = "domyślna" if partia is None else str(partia)
        print(f"{etykieta:>13} {tps:>9,.0f} {CEL/tps/3600:>13.1f} h")
        del nlp
        import torch, gc; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f"{str(partia):>13}   błąd: {str(e)[:40]}")

if wyniki:
    PARTIA_MODELU = max(wyniki, key=wyniki.get)
    tps = wyniki[PARTIA_MODELU]
    print(f"\nUSTAWIONO PARTIA_MODELU = {PARTIA_MODELU}  ({tps:,.0f} tok/s)")
    print(f"Przebieg {SKALA} ({CEL:,} tokenów) zajmie ~{CEL/tps/3600:.1f} h.".replace(",", " "))
    if not UZYJ_DYSKU and CEL / tps / 3600 > 3:
        print("\nUWAGA: przebieg dłuższy niż 3 h, a zapis idzie do /content,")
        print("które znika razem z sesją. Rozważ SKALA = 'pelna' (włącza Dysk).")

In [ ]:
# 5. Zbieranie
#
# Budżet, ścieżki i rozmiar partii pochodzą z komórek 4 i 4b — tutaj nic
# nie zmieniasz.
#
# `co_ile_raport` steruje też opróżnianiem bufora zapisu na dysk. Gdyby sesja
# padła, plik będzie ucięty w ostatniej partii, ale czytelny: `czytaj_trojki`
# to toleruje i odda wszystko, co zdążyło się zapisać.

import sys
if "/content/projekt" not in sys.path:
    sys.path.insert(0, "/content/projekt")

from backend.pipeline import zbierz

stat = zbierz(
    BUDZET, WYJSCIE,
    gpu=True,
    partia=256,                    # grupowanie zdań — bez wpływu na tempo
    partia_modelu=PARTIA_MODELU,   # partia sieci — TO decyduje o tempie
    co_ile_raport=50_000,
)
stat

In [ ]:
# 6. Baza + pierwsze spojrzenie na sygnał
#
# Ten krok NIE potrzebuje GPU — to czysty SQLite, u mnie 14 s dla 1,6 mln
# trójek. Można go równie dobrze zrobić lokalnie:
#     python scripts/zbuduj_z_korpusu.py data/trojki_30M.tsv.gz
# Wtedy da się iterować — zmienić próg, odrzucić źródło — bez limitu sesji,
# a skrypt od razu ocenia sygnał szerzej niż ta komórka.
#
# `postep=print` jest istotne: przy milionach trójek etapy trwają minuty
# i bez raportowania budowa jest nie do odróżnienia od zawieszenia.

from backend.baza import BazaKolokacji, zbuduj
from backend.pipeline import czytaj_trojki

print(zbuduj(czytaj_trojki(WYJSCIE), BAZA, min_pary=3, postep=print))

with BazaKolokacji(BAZA) as db:
    print(db.statystyki())
    print(f"udział wiki {db.udzial_zrodla('wiki'):.1%}, web {db.udzial_zrodla('web'):.1%}")
    print()
    print("Czasowniki łączące się z 'decyzja' (obj:acc):")
    for k in db.alternatywy("obj:acc", "decyzja", limit=10):
        print(f"  {k.lemat:<16} f={k.f:<5} logDice={k.logdice:.2f}")
    print()
    # Przy 5 mln: decyzja 405 (zbadana), porażka 33 (za mało).
    # Przy 30 mln porażka powinna przekroczyć próg — to sprawdzian budżetu.
    for dep in ["decyzja", "porażka", "klęska"]:
        f = db.czestosc_slotu_dep("obj:acc", dep)
        print(f"  {dep:<10} f_brzegowa={f:<6} "
              f"{'ZBADANY' if db.slot_zbadany('obj:acc', dep) else 'za mało — milczymy'}")

In [ ]:
# 7. Kontrola skażenia domenowego
#
# Para o wysokiej częstości pochodząca wyłącznie z webu to zwykle boilerplate
# (stopka, klauzula, szablon ogłoszenia), a nie norma językowa.

with BazaKolokacji(BAZA) as db:
    print("Pary wyłącznie z webu, najczęstsze:")
    for h, s, d, f in db.con.execute(
        """SELECT p.head, p.slot, p.dep, p.f FROM pary p
           WHERE p.f >= 10 AND NOT EXISTS (
             SELECT 1 FROM pary_zrodlo z WHERE z.head=p.head AND z.slot=p.slot
             AND z.dep=p.dep AND z.zrodlo='wiki')
           ORDER BY p.f DESC LIMIT 20"""):
        print(f"  {h} --{s}--> {d}   f={f}")

In [ ]:
# 8. Pobranie wyników na dysk lokalny
#
# Potrzebne tylko przy UZYJ_DYSKU = False. Colab kasuje pliki po rozłączeniu
# sesji, więc bazę trzeba ściągnąć, zanim to nastąpi.

if not UZYJ_DYSKU:
    from google.colab import files
    for plik in [BAZA, WYJSCIE]:
        print(f"{plik.name}: {plik.stat().st_size / 1024 / 1024:.1f} MB")
        files.download(str(plik))
else:
    print(f"Pliki są już na Dysku: {KATALOG}")